In [0]:
from pyspark.sql.functions import current_timestamp, col
base = "/Volumes/workspace/landing/raw"
tables = [("mysql","customers"),("mysql","orders"),("teradata","transactions"),
          ("saphana","gl_postings"),("salesforce","opportunities"),("pos","pos_sales")]

def ingest(source, table):
    (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaLocation", f"{base}/_schemas/{table}")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # schema drift
      .load(f"{base}/{source}/{table}/")
      .withColumn("_ingest_ts", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
      .withColumn("_source_system", __import__("pyspark.sql.functions").sql.functions.lit(source))
      .writeStream
      .option("checkpointLocation", f"{base}/_checkpoints/{table}")
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .toTable(f"workspace.bronze.raw_{table}")
      .awaitTermination())

for s, t in tables:
    ingest(s, t)

In [0]:
%sql
SELECT count(*) FROM workspace.bronze.raw_orders

count(*)
20000
